# Akili Real-Repository Coding Agent V1 — Frozen Held-Out Evaluation

This notebook preserves the reviewed development implementation and switches only the scientific phase to held-out seeds 4, 5 and 6. Do not edit prompts, tests, budgets, templates, scorer or model.


In [ ]:
# FROZEN HELD-OUT LAUNCHER — phase switch only.
import os
os.environ["AKILI_REPO_PHASE"] = "heldout"
for _key in ("AKILI_SESSION", "AKILI_REPO_OUTPUT"):
    os.environ.pop(_key, None)
print("FROZEN REAL-REPOSITORY HELD-OUT: seeds 4, 5, 6 | no parameter overrides")


# Akili Real Repository Coding Agent Colab

**Protocol:** `akili-real-repo-agent-v1`

This notebook is phase locked:

- `development` → seeds 1, 2, 3
- `heldout` → seeds 4, 5, 6

Run development first. Freeze the implementation and criteria before setting the held-out phase. The notebook embeds the extracted V3.1.1a `AkiliCore`; it does not edit the frozen V3.1.1a evidence files.

## Product question
Can Akili restore and supersede repository conventions across two multi-file Python repositories while generated functions are judged by hidden unit tests?

## Arms
Stateless, bounded ICL, and full Akili.


In [ ]:
# CONFIG — environment-variable overrides, no manual seed editing.
import os, sys, json, subprocess, hashlib
from pathlib import Path

PHASE = os.environ.get("AKILI_REPO_PHASE", "development").strip().lower()
LOCKED = {"development": [1, 2, 3], "heldout": [4, 5, 6]}
assert PHASE in LOCKED, "phase must be development or heldout"
SEEDS = LOCKED[PHASE]
MODEL = os.environ.get("AKILI_MODEL", "Qwen/Qwen3-4B")
DRIVE_ROOT = os.environ.get("AKILI_DRIVE_ROOT", "/content/drive/MyDrive/AKILI_CL")
SESSION = os.environ.get("AKILI_SESSION", f"akili_real_repo_agent_{PHASE}_seeds_" + "_".join(map(str, SEEDS)))
WORK_ROOT = Path(os.environ.get("AKILI_WORK_ROOT", "/content/akili_real_repo_agent"))
OUT = Path(DRIVE_ROOT) / SESSION
print({"protocol": "akili-real-repo-agent-v1", "phase": PHASE, "seeds": SEEDS, "model": MODEL, "session": SESSION})


In [ ]:
# Install only the pinned packages required for 4-bit Qwen inference.
subprocess.run([
    sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
    "transformers==4.53.2", "accelerate==1.8.1", "bitsandbytes==0.46.1"
], check=True, timeout=480)


In [ ]:
# Mount Drive and validate CUDA before downloading the model.
from google.colab import drive
drive.mount("/content/drive")
import torch
assert torch.cuda.is_available(), "CUDA GPU required — stop before model download"
WORK_ROOT.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)
print("WORK_ROOT:", WORK_ROOT)
print("OUT:", OUT)
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# Embedded V3.1.1a core source — written verbatim for import.
CORE_SOURCE = '# =============================================================================\n# V3.1.1 SYSTEMS — independent instances, bounded ICL state, scoped Akili state.\n# =============================================================================\nimport copy\nimport hashlib\nimport json\nimport re\nimport uuid\n\n\ndef truncate_to_budget(text, budget_tokens, count_fn):\n    text = text or ""\n    if budget_tokens <= 0:\n        return "", 0\n    if count_fn(text) <= budget_tokens:\n        return text, count_fn(text)\n    lo, hi = 0, len(text)\n    while lo < hi:\n        mid = (lo + hi + 1) // 2\n        if count_fn(text[:mid]) <= budget_tokens:\n            lo = mid\n        else:\n            hi = mid - 1\n    fitted = text[:lo]\n    used = count_fn(fitted)\n    while fitted and used > budget_tokens:\n        fitted = fitted[:-1]\n        used = count_fn(fitted)\n    return fitted, used\n\n\ndef fit_newest_to_budget(parts_newest_first, budget_tokens, count_fn, header=""):\n    chosen = []\n    base = header\n    for part in parts_newest_first:\n        candidate_parts = chosen + [part]\n        body = "\\n\\n".join(candidate_parts)\n        candidate = (base + "\\n" + body).strip() if base else body\n        if count_fn(candidate) <= budget_tokens:\n            chosen.append(part)\n            continue\n        remaining = budget_tokens - count_fn((base + "\\n" + "\\n\\n".join(chosen)).strip())\n        if remaining > 0:\n            fitted, _ = truncate_to_budget(part, remaining, count_fn)\n            if fitted:\n                chosen.append(fitted)\n        break\n    text = ((base + "\\n") if base else "") + "\\n\\n".join(chosen)\n    text = text.strip()\n    text, used = truncate_to_budget(text, budget_tokens, count_fn)\n    return text, used\n\n\nclass Stateless:\n    name = "stateless"\n\n    def __init__(self, count_fn):\n        self.count = count_fn\n        self.instance_id = uuid.uuid4().hex\n\n    def memory_prompt(self):\n        return ""\n\n    def observe_episode(self, summary, features, entity_hint, outcome):\n        return None\n\n    def report(self):\n        return {\n            "instance_id": self.instance_id,\n            "stored_memory_chars": 0,\n            "stored_memory_tokens": 0,\n            "active_rules": 0,\n            "admitted_total": 0,\n            "superseded": 0,\n        }\n\n\nclass BoundedICL:\n    name = "bounded_icl"\n\n    def __init__(self, budget_tokens, count_fn):\n        self.budget = budget_tokens\n        self.count = count_fn\n        self.history = []\n        self.instance_id = uuid.uuid4().hex\n\n    def _render(self):\n        if not self.history:\n            return "", 0\n        return fit_newest_to_budget(\n            list(reversed(self.history)), self.budget, self.count,\n            header="RECENT RAW EPISODE HISTORY:",\n        )\n\n    def _trim_state(self):\n        while len(self.history) > 1:\n            blob, used = self._render()\n            if used <= self.budget and self.count("\\n\\n".join(self.history)) <= self.budget:\n                break\n            self.history.pop(0)\n        if self.history:\n            latest, _ = truncate_to_budget(self.history[-1], self.budget, self.count)\n            self.history[-1] = latest\n\n    def memory_prompt(self):\n        return self._render()[0]\n\n    def observe_episode(self, summary, features, entity_hint, outcome):\n        self.history.append(str(summary))\n        self._trim_state()\n\n    def report(self):\n        blob = "\\n\\n".join(self.history)\n        prompt, prompt_tokens = self._render()\n        return {\n            "instance_id": self.instance_id,\n            "history_entries": len(self.history),\n            "stored_memory_chars": len(blob),\n            "stored_memory_tokens": self.count(blob),\n            "retrievable_tokens": prompt_tokens,\n            "state_bounded": self.count(blob) <= self.budget,\n            "active_rules": 0,\n            "admitted_total": 0,\n            "superseded": 0,\n        }\n\n\n_NAME_OK = re.compile(r"^[A-Za-z][A-Za-z0-9 .\'\\-_]{0,31}$")\n_NAME_BAD_WORDS = frozenset({"policy", "stage", "schedule", "instance"})\n\n\n\nclass AkiliCore:\n    name = "akili"\n\n    def __init__(self, budget_tokens, count_fn, derive_fn, min_evidence=2,\n                 max_profiles=16, max_versions=48, recent_window=10,\n                 max_stats_keys=96, max_retrieval_log=512):\n        self.budget = budget_tokens\n        self.count = count_fn\n        self.derive_fn = derive_fn\n        self.min_evidence = min_evidence\n        self.max_profiles = max_profiles\n        self.max_versions = max_versions\n        self.recent_window = recent_window\n        self.max_stats_keys = max_stats_keys\n        self.max_retrieval_log = max_retrieval_log\n        self.instance_id = uuid.uuid4().hex\n        self.profiles = {}\n        self.current_key = None\n        self.audit = []\n        self.episode_count = 0\n        self.cross_scope_retrievals = 0\n        self.retrieval_log = []\n        self._audit("RUN_START", {})\n\n    def _audit(self, event, payload):\n        body = {\n            "seq": len(self.audit), "event": event, "episode": self.episode_count,\n            "profile": self.current_key, "payload": payload,\n            "prev_hash": self.audit[-1]["hash"] if self.audit else "GENESIS",\n        }\n        body["hash"] = hashlib.sha256(\n            json.dumps(body, sort_keys=True, default=str).encode("utf-8")\n        ).hexdigest()\n        self.audit.append(body)\n\n    def validate_chain(self):\n        prev = "GENESIS"\n        for entry in self.audit:\n            body = {k: v for k, v in entry.items() if k != "hash"}\n            if body["prev_hash"] != prev:\n                return False\n            expected = hashlib.sha256(\n                json.dumps(body, sort_keys=True, default=str).encode("utf-8")\n            ).hexdigest()\n            if expected != entry["hash"]:\n                return False\n            prev = entry["hash"]\n        return True\n\n    def _new_profile(self, key, display):\n        assert len(self.profiles) < self.max_profiles, "Akili profile bound exceeded"\n        self.profiles[key] = {\n            "key": key, "display": display, "status": "ACTIVE", "episodes": 0,\n            "stats": {}, "recent_features": [], "versions": [], "version_seq": 0,\n            "active_by_family": {}, "times_activated": 1,\n        }\n\n    def _accept_name(self, raw):\n        name = str(raw or "").strip()\n        if not _NAME_OK.match(name):\n            return None\n        if any(word in name.lower() for word in _NAME_BAD_WORDS):\n            return None\n        return name\n\n    def switch_entity(self, raw_name):\n        name = self._accept_name(raw_name)\n        if name is None:\n            key = "anon:" + hashlib.sha256(str(raw_name).encode()).hexdigest()[:16]\n            display = None\n        else:\n            key = "name:" + hashlib.sha256(name.lower().encode()).hexdigest()[:16]\n            display = name\n        if key == self.current_key:\n            return\n        previous = self.current_key\n        if previous is not None:\n            self.profiles[previous]["status"] = "DORMANT"\n        if key in self.profiles:\n            self.profiles[key]["status"] = "ACTIVE"\n            self.profiles[key]["times_activated"] += 1\n            self.current_key = key\n            self._audit("PROFILE_REACTIVATED", {"from": previous, "to": key, "display": display})\n        else:\n            self._new_profile(key, display)\n            self.current_key = key\n            self._audit("PROFILE_CREATED", {"from": previous, "to": key, "display": display})\n\n    def _append_version(self, profile, version):\n        if len(profile["versions"]) >= self.max_versions:\n            removable = next((i for i, item in enumerate(profile["versions"])\n                              if item["status"] in {"REJECTED", "SUPERSEDED"}), None)\n            assert removable is not None, "Akili version bound exceeded with no removable record"\n            removed = profile["versions"].pop(removable)\n            self._audit("VERSION_EVICTED", {"version_hash": removed["hash"], "status": removed["status"]})\n        profile["versions"].append(version)\n\n    def observe_episode(self, summary, features, entity_hint, outcome):\n        if self.current_key is None:\n            self.switch_entity(entity_hint or "unnamed")\n        self.episode_count += 1\n        profile = self.profiles[self.current_key]\n        profile["episodes"] += 1\n        features = dict(features or {})\n        profile["recent_features"].append(features)\n        profile["recent_features"] = profile["recent_features"][-self.recent_window:]\n        for key, value in features.items():\n            if key not in profile["stats"]:\n                assert len(profile["stats"]) < self.max_stats_keys, "Akili stats-key bound exceeded"\n            if isinstance(value, (int, float)):\n                profile["stats"][key] = profile["stats"].get(key, 0) + value\n            elif key.startswith("latest_"):\n                profile["stats"][key] = value\n            else:\n                profile["stats"].setdefault(key, value)\n        self._audit("EPISODE_RECORDED", {"features": features, "outcome": outcome})\n        candidates = self.derive_fn(profile["stats"], profile, features)\n        if not candidates:\n            return\n        if isinstance(candidates, dict):\n            candidates = [candidates]\n        for candidate in candidates:\n            self._process_candidate(profile, candidate)\n\n    @staticmethod\n    def _version_hash(version):\n        body = {k: v for k, v in version.items() if k != "hash"}\n        return hashlib.sha256(json.dumps(body, sort_keys=True, default=str).encode("utf-8")).hexdigest()\n\n    def _process_candidate(self, profile, candidate):\n        family = candidate.get("family", "general")\n        lifecycle = candidate.get("lifecycle_state", "VERIFIED")\n        assert lifecycle in {"PROVISIONAL", "VERIFIED", "CONFLICTED"}\n        active = profile["active_by_family"].get(family)\n        same_rule = active is not None and active["rule"] == candidate["rule"]\n        same_state = active is not None and active.get("lifecycle_state", active["status"]) == lifecycle\n\n        schema_template = candidate.get("schema_template")\n        template_metadata = copy.deepcopy(candidate.get("template_metadata") or {})\n        template_hash = hashlib.sha256(schema_template.encode("utf-8")).hexdigest() if schema_template else None\n        verified_exemplar = candidate.get("verified_exemplar") or candidate.get("exemplar")\n        verified_structural_exemplar = candidate.get("verified_structural_exemplar")\n        exemplar_shape_key = candidate.get("exemplar_shape_key")\n\n        if same_rule and same_state:\n            changed = False\n            old_hash = active["hash"]\n            if schema_template and schema_template != active.get("schema_template"):\n                active["schema_template"] = schema_template\n                active["schema_template_hash"] = template_hash\n                active["template_metadata"] = template_metadata\n                changed = True\n                self._audit("SCHEMA_TEMPLATE_ADDED", {\n                    "family": family, "template_hash": template_hash,\n                    "shape_key": template_metadata.get("shape_key"),\n                    "source": template_metadata.get("source"),\n                })\n            if verified_exemplar and exemplar_shape_key:\n                exemplars = active.setdefault("verified_exemplars", {})\n                previous = exemplars.get(exemplar_shape_key)\n                exemplar_hash = hashlib.sha256(verified_exemplar.encode("utf-8")).hexdigest()\n                if not previous or previous.get("hash") != exemplar_hash:\n                    structural_hash = (\n                        hashlib.sha256(verified_structural_exemplar.encode("utf-8")).hexdigest()\n                        if verified_structural_exemplar else None\n                    )\n                    exemplars[exemplar_shape_key] = {\n                        "content": verified_exemplar,\n                        "hash": exemplar_hash,\n                        "structural_content": verified_structural_exemplar,\n                        "structural_hash": structural_hash,\n                        "shape_key": exemplar_shape_key,\n                        "verified": True,\n                    }\n                    changed = True\n                    self._audit("VERIFIED_EXEMPLAR_ADDED", {\n                        "family": family, "shape_key": exemplar_shape_key,\n                        "exemplar_hash": exemplar_hash,\n                    })\n            if changed:\n                active["hash"] = self._version_hash(active)\n                self._audit("ACTIVE_RECORD_UPDATED", {\n                    "family": family, "old_hash": old_hash, "new_hash": active["hash"],\n                })\n            else:\n                self._audit("CANDIDATE_DUPLICATE", {\n                    "family": family, "lifecycle_state": lifecycle,\n                    "evidence": candidate.get("evidence", {}),\n                })\n            return\n\n        min_gap = int(candidate.get("min_transition_gap", 0))\n        if active is not None and profile["episodes"] - active["episode"] < min_gap:\n            self._audit("TRANSITION_DEFERRED", {\n                "family": family, "lifecycle_state": lifecycle,\n                "episodes_since_active": profile["episodes"] - active["episode"],\n                "evidence": candidate.get("evidence", {}),\n            })\n            return\n\n        audit_candidate = {\n            k: v for k, v in candidate.items()\n            if k not in {"schema_template", "verified_exemplar", "verified_structural_exemplar", "exemplar"}\n        }\n        audit_candidate["schema_template_hash"] = template_hash\n        if verified_exemplar:\n            audit_candidate["verified_exemplar_hash"] = hashlib.sha256(verified_exemplar.encode("utf-8")).hexdigest()\n        if verified_structural_exemplar:\n            audit_candidate["verified_structural_exemplar_hash"] = hashlib.sha256(\n                verified_structural_exemplar.encode("utf-8")\n            ).hexdigest()\n        self._audit("CANDIDATE_CREATED", audit_candidate)\n        admitted, reason = self._gate(candidate)\n        profile["version_seq"] += 1\n        verified_exemplars = {}\n        if verified_exemplar and exemplar_shape_key:\n            exemplar_hash = hashlib.sha256(verified_exemplar.encode("utf-8")).hexdigest()\n            structural_hash = (\n                hashlib.sha256(verified_structural_exemplar.encode("utf-8")).hexdigest()\n                if verified_structural_exemplar else None\n            )\n            verified_exemplars[exemplar_shape_key] = {\n                "content": verified_exemplar,\n                "hash": exemplar_hash,\n                "structural_content": verified_structural_exemplar,\n                "structural_hash": structural_hash,\n                "shape_key": exemplar_shape_key,\n                "verified": True,\n            }\n        version = {\n            "version": profile["version_seq"], "family": family, "rule": candidate["rule"],\n            "evidence": candidate.get("evidence", {}),\n            "status": lifecycle if admitted else "REJECTED",\n            "lifecycle_state": lifecycle if admitted else "REJECTED",\n            "gate_reason": reason, "episode": profile["episodes"],\n            "decision_label": candidate.get("decision_label"),\n            "schema_template": schema_template,\n            "schema_template_hash": template_hash,\n            "template_metadata": template_metadata,\n            "verified_exemplars": verified_exemplars,\n        }\n        version["hash"] = self._version_hash(version)\n        self._append_version(profile, version)\n        if admitted:\n            if active is not None:\n                old_state = active.get("lifecycle_state", active.get("status"))\n                active["status"] = "SUPERSEDED"\n                self._audit("SUPERSEDED", {\n                    "family": family, "old_rule": active["rule"], "new_rule": candidate["rule"],\n                    "old_state": old_state, "new_state": lifecycle,\n                    "evidence": candidate.get("evidence", {}),\n                })\n            profile["active_by_family"][family] = version\n            self._audit("ADMITTED", {\n                "family": family, "rule": candidate["rule"], "lifecycle_state": lifecycle,\n                "decision_label": candidate.get("decision_label"),\n                "evidence": candidate.get("evidence", {}), "reason": reason,\n                "schema_template_hash": template_hash,\n                "template_source": template_metadata.get("source"),\n                "template_shape_key": template_metadata.get("shape_key"),\n                "verified_exemplar_hashes": {\n                    key: value["hash"] for key, value in verified_exemplars.items()\n                },\n            })\n        else:\n            self._audit("REJECTED", {"family": family, "evidence": candidate.get("evidence", {}), "reason": reason})\n\n    def _gate(self, candidate):\n        evidence = candidate.get("evidence", {})\n        count = int(evidence.get("count", 0))\n        threshold = int(candidate.get("admit_threshold", self.min_evidence))\n        if count < threshold:\n            return False, f"evidence {count} < {threshold}"\n        return True, f"evidence {count} >= {threshold}"\n\n    def active_record(self, family):\n        if self.current_key is None:\n            return None\n        return self.profiles[self.current_key]["active_by_family"].get(family)\n\n    def retrieval_context(self, family=None, operation_shape=None):\n        details = {\n            "requested_family": family,\n            "operation_shape": operation_shape,\n            "rule_available": False,\n            "persistent_rule_available": False,\n            "rule_retrieved": False,\n            "schema_template_available": False,\n            "schema_template_retrieved": False,\n            "persistent_schema_template_retrieved": False,\n            "current_instruction_template_available": False,\n            "current_instruction_template_retrieved": False,\n            "verified_exemplar_available": False,\n            "verified_exemplar_retrieved": False,\n            "verified_structural_exemplar_available": False,\n            "verified_structural_exemplar_retrieved": False,\n            "raw_verified_exemplar_retrieved": False,\n            "verified_exemplar_suppressed_by_template": False,\n            "shape_match": False,\n            "memory_provenance": "NONE",\n            "primary_memory_class": "NO_MEMORY",\n            "applicable_rule_state": None,\n            "applicable_decision_label": None,\n            "applicable_rule_hash": None,\n            "template_hash_at_decision": None,\n            "template_source_at_decision": None,\n            "prior_active_convention_suppressed": False,\n        }\n        if self.current_key is None:\n            return "", details\n        profile = self.profiles[self.current_key]\n        if family is None:\n            actives = list(profile["active_by_family"].values())\n        else:\n            active = profile["active_by_family"].get(family)\n            actives = [active] if active is not None else []\n        if not actives:\n            self.retrieval_log.append({\n                "episode": self.episode_count, "requested": self.current_key,\n                "retrieved": self.current_key, "tokens": 0, **details,\n            })\n            self.retrieval_log = self.retrieval_log[-self.max_retrieval_log:]\n            self._audit("PROFILE_RETRIEVED", {"retrieved": self.current_key, "tokens": 0, **details})\n            return "", details\n\n        lines = [f"=== AKILI PROFILE: {profile[\'display\'] or \'unnamed\'} | episodes={profile[\'episodes\']} ==="]\n        provenance_rank = {\n            "NONE": 0, "RULE": 1, "RAW_VERIFIED_EXEMPLAR": 2,\n            "VERIFIED_STRUCTURAL_EXEMPLAR": 3, "SCHEMA_TEMPLATE": 4,\n        }\n        for item in actives:\n            state = item.get("lifecycle_state", item["status"])\n            lines.append(f"- [{item[\'family\']} | {state}] {item[\'rule\']}")\n            details["rule_available"] = True\n            details["persistent_rule_available"] = True\n            details["rule_retrieved"] = True\n            details["applicable_rule_state"] = state if family else details["applicable_rule_state"]\n            details["applicable_decision_label"] = item.get("decision_label") if family else details["applicable_decision_label"]\n            details["applicable_rule_hash"] = item.get("hash") if family else details["applicable_rule_hash"]\n            if provenance_rank["RULE"] > provenance_rank.get(details["memory_provenance"], 0):\n                details["memory_provenance"] = "RULE"\n                details["primary_memory_class"] = "RULE_ONLY"\n            ev = item.get("evidence") or {}\n            compact = {k: ev[k] for k in (\n                "malicious_count", "benign_count", "recent_labels", "count", "signature"\n            ) if k in ev}\n            if compact:\n                lines.append("  Evidence: " + json.dumps(compact, sort_keys=True))\n\n            exemplars = item.get("verified_exemplars") or {}\n            exact_exemplar = exemplars.get(operation_shape) if operation_shape else None\n            structural = (exact_exemplar or {}).get("structural_content") if exact_exemplar else None\n            details["verified_exemplar_available"] = bool(exact_exemplar)\n            details["verified_structural_exemplar_available"] = bool(structural)\n            template = item.get("schema_template")\n            template_meta = item.get("template_metadata") or {}\n            template_matches = bool(template) and (\n                operation_shape is None or template_meta.get("shape_key") == operation_shape\n            )\n            details["schema_template_available"] = bool(template)\n            details["shape_match"] = bool(exact_exemplar or template_matches)\n\n            # V3.1.1 precedence: a value-free template outranks any concrete prior output.\n            if template_matches:\n                lines.append(\n                    "  MEMORY TYPE: SCHEMA_TEMPLATE (persistent, deterministic, value-free, not scorer-verified; "\n                    "replace every symbolic placeholder with exact values from the CURRENT issue brief)\\n"\n                    + template.strip()\n                )\n                details["schema_template_retrieved"] = True\n                details["persistent_schema_template_retrieved"] = True\n                details["template_hash_at_decision"] = item.get("schema_template_hash")\n                details["template_source_at_decision"] = template_meta.get("source")\n                details["memory_provenance"] = "SCHEMA_TEMPLATE"\n                details["primary_memory_class"] = "PERSISTENT_SCHEMA_TEMPLATE"\n                details["verified_exemplar_suppressed_by_template"] = bool(exact_exemplar)\n            elif structural:\n                lines.append(\n                    "  MEMORY TYPE: VERIFIED_STRUCTURAL_EXEMPLAR (derived from a scorer-passing implementation, "\n                    "but value-free; replace placeholders with CURRENT issue values)\\n"\n                    + structural.strip()\n                )\n                details["verified_exemplar_retrieved"] = True\n                details["verified_structural_exemplar_retrieved"] = True\n                details["memory_provenance"] = "VERIFIED_STRUCTURAL_EXEMPLAR"\n                details["primary_memory_class"] = "VERIFIED_STRUCTURAL_EXEMPLAR"\n            elif exact_exemplar:\n                lines.append(\n                    "  MEMORY TYPE: RAW_VERIFIED_EXEMPLAR (scorer-passing historical output; use only its structure "\n                    "and replace all values from the CURRENT issue brief)\\n"\n                    + exact_exemplar["content"].strip()\n                )\n                details["verified_exemplar_retrieved"] = True\n                details["raw_verified_exemplar_retrieved"] = True\n                details["memory_provenance"] = "RAW_VERIFIED_EXEMPLAR"\n                details["primary_memory_class"] = "RAW_VERIFIED_EXEMPLAR"\n        lines.append("=== END AKILI PROFILE ===")\n        block, used = truncate_to_budget("\\n".join(lines), self.budget, self.count)\n        retrieved_key = profile["key"]\n        if retrieved_key != self.current_key:\n            self.cross_scope_retrievals += 1\n        row = {\n            "episode": self.episode_count, "requested": self.current_key,\n            "retrieved": retrieved_key, "tokens": used, **details,\n        }\n        self.retrieval_log.append(row)\n        self.retrieval_log = self.retrieval_log[-self.max_retrieval_log:]\n        self._audit("PROFILE_RETRIEVED", {"retrieved": retrieved_key, "tokens": used, **details})\n        return block, details\n\n    def memory_prompt(self):\n        return self.retrieval_context()[0]\n\n    def snapshot(self):\n        return {\n            "instance_id": self.instance_id, "current_key": self.current_key,\n            "profiles": copy.deepcopy(self.profiles), "retrieval_log": copy.deepcopy(self.retrieval_log),\n        }\n\n    def report(self):\n        live = {"PROVISIONAL", "VERIFIED", "CONFLICTED"}\n        admitted_total = sum(1 for p in self.profiles.values() for v in p["versions"]\n                             if v["status"] in live | {"SUPERSEDED"})\n        superseded = sum(1 for p in self.profiles.values() for v in p["versions"] if v["status"] == "SUPERSEDED")\n        active_rules = sum(len(p["active_by_family"]) for p in self.profiles.values())\n        lifecycle_counts = {state: 0 for state in ["PROVISIONAL", "VERIFIED", "CONFLICTED", "REJECTED"]}\n        status_counts = {state: 0 for state in ["PROVISIONAL", "VERIFIED", "CONFLICTED", "SUPERSEDED", "REJECTED"]}\n        schema_templates = 0\n        rejected_schema_templates = 0\n        verified_exemplars = 0\n        for p in self.profiles.values():\n            for v in p["versions"]:\n                lifecycle = v.get("lifecycle_state", v["status"])\n                lifecycle_counts[lifecycle] = lifecycle_counts.get(lifecycle, 0) + 1\n                status_counts[v["status"]] = status_counts.get(v["status"], 0) + 1\n                if v.get("status") == "REJECTED":\n                    rejected_schema_templates += int(bool(v.get("schema_template")))\n                else:\n                    schema_templates += int(bool(v.get("schema_template")))\n                    verified_exemplars += len(v.get("verified_exemplars") or {})\n        state_blob = json.dumps(self.snapshot(), sort_keys=True, default=str)\n        state_bounded = (\n            len(self.profiles) <= self.max_profiles and len(self.retrieval_log) <= self.max_retrieval_log\n            and all(len(p["versions"]) <= self.max_versions for p in self.profiles.values())\n            and all(len(p["stats"]) <= self.max_stats_keys for p in self.profiles.values())\n        )\n        return {\n            "instance_id": self.instance_id, "profiles": len(self.profiles),\n            "stored_memory_chars": len(state_blob), "stored_memory_tokens": self.count(state_blob),\n            "active_rules": active_rules, "admitted_total": admitted_total, "superseded": superseded,\n            "lifecycle_counts": lifecycle_counts, "status_counts": status_counts,\n            "schema_templates": schema_templates,\n            "rejected_schema_templates": rejected_schema_templates,\n            "verified_exemplars": verified_exemplars,\n            "audit_events": len(self.audit), "audit_valid": self.validate_chain(),\n            "cross_scope_retrievals": self.cross_scope_retrievals, "state_bounded": state_bounded,\n            "current_entity": self.current_key,\n        }\n'
core_path = WORK_ROOT / "akili_v311a_core.py"
core_path.write_text(CORE_SOURCE, encoding="utf-8")
print("core sha256:", hashlib.sha256(core_path.read_bytes()).hexdigest())


In [ ]:
# Embedded benchmark source.
BENCHMARK_SOURCE = '#!/usr/bin/env python3\n"""Akili real-repository coding-agent benchmark.\n\nCreates two small, multi-file Python repositories, applies model-generated\nfunctions, and validates them with hidden unittest suites in isolated temporary\ncopies. The public issue prompt never contains the tests.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport ast\nimport hashlib\nimport json\nimport os\nimport random\nimport re\nimport shutil\nimport subprocess\nimport sys\nimport tempfile\nfrom collections import defaultdict\nfrom pathlib import Path\nfrom typing import Any, Dict, Iterable, List, Optional, Tuple\n\nfrom akili_v311a_core import AkiliCore, BoundedICL, Stateless, truncate_to_budget\n\nPROTOCOL = "akili-real-repo-agent-v1"\nLOCKED_PHASE_SEEDS = {"development": [1, 2, 3], "heldout": [4, 5, 6]}\nARMS = ["stateless", "bounded_icl", "akili"]\n\n\ndef sha256_bytes(data: bytes) -> str:\n    return hashlib.sha256(data).hexdigest()\n\n\ndef canonical_hash(obj: Any) -> str:\n    return sha256_bytes(json.dumps(obj, sort_keys=True, default=str).encode())\n\n\ndef token_count_approx(text: str) -> int:\n    return max(0, (len(text or "") + 3) // 4)\n\n\nFAMILY_META = {\n    "solar_voltage_validation": {\n        "repo": "solar_ops", "module": "src/solar_ops/validators.py", "function": "validate_voltage",\n        "shape": "unary_voltage_validation",\n        "template": \'\'\'def validate_voltage(value):\n    if value < <LOWER_BOUND> or value > <UPPER_BOUND>:\n        raise <EXCEPTION_TYPE>(<MESSAGE>)\n    return value\'\'\',\n    },\n    "solar_config_precedence": {\n        "repo": "solar_ops", "module": "src/solar_ops/config.py", "function": "resolve_setting",\n        "shape": "first_non_null_setting",\n        "template": \'\'\'def resolve_setting(cli, env, file_value, default):\n    for value in (<SOURCE_1>, <SOURCE_2>, <SOURCE_3>, <SOURCE_4>):\n        if value is not None:\n            return value\n    return None\'\'\',\n    },\n    "field_record_serialization": {\n        "repo": "field_data", "module": "src/field_data/records.py", "function": "serialize_observation",\n        "shape": "observation_to_dict",\n        "template": \'\'\'def serialize_observation(record):\n    return {\n        <OUTPUT_KEY_1>: record[<INPUT_KEY_1>],\n        <OUTPUT_KEY_2>: record[<INPUT_KEY_2>],\n    }\'\'\',\n    },\n    "field_artifact_naming": {\n        "repo": "field_data", "module": "src/field_data/paths.py", "function": "artifact_path",\n        "shape": "artifact_path_naming",\n        "template": \'\'\'def artifact_path(root, project, name):\n    return f"{root}/{project}/<PREFIX>_{name}.<EXTENSION>"\'\'\',\n    },\n}\nFAMILIES = list(FAMILY_META)\n\n\ndef params_for(family: str, version: int, seed: int) -> Dict[str, Any]:\n    rng = random.Random(seed * 10_000 + FAMILIES.index(family) * 100 + version)\n    if family == "solar_voltage_validation":\n        low = rng.choice([0, 10, 50]) + version\n        return {"low": low, "high": low + rng.choice([100, 200, 400]), "exception": "ValueError" if version == 1 else "RuntimeError", "message": f"voltage-policy-v{version}"}\n    if family == "solar_config_precedence":\n        return {"order": ["cli", "env", "file_value", "default"] if version == 1 else ["env", "file_value", "cli", "default"]}\n    if family == "field_record_serialization":\n        return {"in1": f"asset_v{version}", "in2": f"reading_v{version}", "out1": f"asset_id_v{version}", "out2": f"measurement_v{version}"}\n    if family == "field_artifact_naming":\n        return {"prefix": "inspection" if version == 1 else "verified", "ext": ["json", "txt", "dat"][seed % 3]}\n    raise KeyError(family)\n\n\ndef public_rule(family: str, p: Dict[str, Any]) -> str:\n    if family == "solar_voltage_validation":\n        return f"accept inclusive voltage range [{p[\'low\']}, {p[\'high\']}]; otherwise raise {p[\'exception\']} with {p[\'message\']!r}"\n    if family == "solar_config_precedence":\n        return "setting precedence is " + " > ".join(p["order"])\n    if family == "field_record_serialization":\n        return f"map {p[\'in1\']!r},{p[\'in2\']!r} to {p[\'out1\']!r},{p[\'out2\']!r}"\n    if family == "field_artifact_naming":\n        return f"artifact path uses {p[\'prefix\']}_<name>.{p[\'ext\']}"\n    raise KeyError(family)\n\n\ndef expected_code(family: str, p: Dict[str, Any]) -> str:\n    fn = FAMILY_META[family]["function"]\n    if family == "solar_voltage_validation":\n        return f\'\'\'def {fn}(value):\n    if value < {p["low"]!r} or value > {p["high"]!r}:\n        raise {p["exception"]}({p["message"]!r})\n    return value\n\'\'\'\n    if family == "solar_config_precedence":\n        return f\'\'\'def {fn}(cli, env, file_value, default):\n    for value in ({", ".join(p["order"])}):\n        if value is not None:\n            return value\n    return None\n\'\'\'\n    if family == "field_record_serialization":\n        return f\'\'\'def {fn}(record):\n    return {{\n        {p["out1"]!r}: record[{p["in1"]!r}],\n        {p["out2"]!r}: record[{p["in2"]!r}],\n    }}\n\'\'\'\n    if family == "field_artifact_naming":\n        return f\'\'\'def {fn}(root, project, name):\n    return f"{{root}}/{{project}}/{p["prefix"]}_{{name}}.{p["ext"]}"\n\'\'\'\n    raise KeyError(family)\n\n\ndef current_template(family: str) -> str:\n    return (\n        "MEMORY TYPE: CURRENT_INSTRUCTION_TEMPLATE\\n"\n        "Source: deterministic renderer from current repository metadata.\\n"\n        "Replace placeholders only with values from the current issue.\\n"\n        + FAMILY_META[family]["template"]\n    )\n\n\ndef make_issues(seed: int) -> List[Dict[str, Any]]:\n    stages = [\n        [(f, 1, True, "acquisition_1") for f in FAMILIES],\n        [(f, 1, True, "acquisition_2") for f in reversed(FAMILIES)],\n        [(f, 1, False, "return_after_repo_switch") for f in FAMILIES],\n        [(f, 2, True, "supersession_1") for f in reversed(FAMILIES)],\n        [(f, 2, True, "supersession_2") for f in FAMILIES],\n        [(f, 2, False, "post_supersession_return") for f in reversed(FAMILIES)],\n    ]\n    rng = random.Random(seed)\n    result: List[Dict[str, Any]] = []\n    issue_id = 0\n    for stage in stages:\n        stage = list(stage)\n        rng.shuffle(stage)\n        for family, version, teach, slice_name in stage:\n            issue_id += 1\n            p = params_for(family, version, seed)\n            meta = FAMILY_META[family]\n            issue = {\n                "id": issue_id, "seed": seed, "family": family, "version": version,\n                "repo": meta["repo"], "module": meta["module"], "function": meta["function"],\n                "shape": meta["shape"], "teach": teach, "slice": slice_name,\n                "params": p, "rule": public_rule(family, p),\n                "expected_code": expected_code(family, p),\n            }\n            result.append(issue)\n    return result\n\n\ndef repo_files(repo: str) -> Dict[str, str]:\n    if repo == "solar_ops":\n        return {\n            "README.md": "# solar_ops\\nSmall solar operations utility package used by the Akili benchmark.\\n",\n            "src/solar_ops/__init__.py": "from .validators import validate_voltage\\nfrom .config import resolve_setting\\n",\n            "src/solar_ops/validators.py": "def validate_voltage(value):\\n    raise NotImplementedError(\'implement\')\\n",\n            "src/solar_ops/config.py": "def resolve_setting(cli, env, file_value, default):\\n    raise NotImplementedError(\'implement\')\\n",\n            "src/solar_ops/metrics.py": "def efficiency(output, input_value):\\n    return output / input_value if input_value else 0.0\\n",\n        }\n    if repo == "field_data":\n        return {\n            "README.md": "# field_data\\nSmall field-data utility package used by the Akili benchmark.\\n",\n            "src/field_data/__init__.py": "from .records import serialize_observation\\nfrom .paths import artifact_path\\n",\n            "src/field_data/records.py": "def serialize_observation(record):\\n    raise NotImplementedError(\'implement\')\\n",\n            "src/field_data/paths.py": "def artifact_path(root, project, name):\\n    raise NotImplementedError(\'implement\')\\n",\n            "src/field_data/quality.py": "def is_complete(record, required):\\n    return all(key in record for key in required)\\n",\n        }\n    raise KeyError(repo)\n\n\ndef repo_context(repo: str) -> str:\n    parts = []\n    for path, text in sorted(repo_files(repo).items()):\n        parts.append(f"--- {path} ---\\n{text}")\n    return "\\n".join(parts)\n\n\ndef hidden_test_source(issue: Dict[str, Any]) -> str:\n    p, module, fn = issue["params"], issue["module"].replace("src/", "").replace(".py", "").replace("/", "."), issue["function"]\n    if issue["family"] == "solar_voltage_validation":\n        return f\'\'\'import unittest\nfrom {module} import {fn}\nclass HiddenTests(unittest.TestCase):\n    def test_range(self):\n        self.assertEqual({fn}({p["low"]}), {p["low"]})\n        self.assertEqual({fn}({p["high"]}), {p["high"]})\n    def test_outside(self):\n        for value in ({p["low"] - 1}, {p["high"] + 1}):\n            with self.assertRaisesRegex({p["exception"]}, {p["message"]!r}):\n                {fn}(value)\nif __name__ == "__main__": unittest.main()\n\'\'\'\n    if issue["family"] == "solar_config_precedence":\n        first, second = p["order"][:2]\n        return f\'\'\'import unittest\nfrom {module} import {fn}\nclass HiddenTests(unittest.TestCase):\n    def test_precedence(self):\n        values = {{"cli":"CLI", "env":"ENV", "file_value":"FILE", "default":"DEF"}}\n        self.assertEqual({fn}(values["cli"], values["env"], values["file_value"], values["default"]), values[{first!r}])\n        values[{first!r}] = None\n        self.assertEqual({fn}(values["cli"], values["env"], values["file_value"], values["default"]), values[{second!r}])\nif __name__ == "__main__": unittest.main()\n\'\'\'\n    if issue["family"] == "field_record_serialization":\n        return f\'\'\'import unittest\nfrom {module} import {fn}\nclass HiddenTests(unittest.TestCase):\n    def test_mapping(self):\n        record = {{{p["in1"]!r}: 12, {p["in2"]!r}: 7.5, "noise": True}}\n        self.assertEqual({fn}(record), {{{p["out1"]!r}: 12, {p["out2"]!r}: 7.5}})\nif __name__ == "__main__": unittest.main()\n\'\'\'\n    if issue["family"] == "field_artifact_naming":\n        return f\'\'\'import unittest\nfrom {module} import {fn}\nclass HiddenTests(unittest.TestCase):\n    def test_path(self):\n        self.assertEqual({fn}("/tmp", "site-a", "panel"), {f"/tmp/site-a/{p[\'prefix\']}_panel.{p[\'ext\']}"!r})\nif __name__ == "__main__": unittest.main()\n\'\'\'\n    raise KeyError(issue["family"])\n\n\nFORBIDDEN = (ast.Import, ast.ImportFrom, ast.While, ast.With, ast.AsyncWith, ast.ClassDef, ast.Lambda, ast.Global, ast.Nonlocal, ast.Delete, ast.Yield, ast.YieldFrom, ast.Await)\n\n\ndef normalize_code(text: str) -> str:\n    text = (text or "").strip()\n    if text.startswith("```"):\n        lines = text.splitlines()[1:]\n        if lines and lines[-1].strip() == "```": lines = lines[:-1]\n        text = "\\n".join(lines).strip()\n    return text\n\n\ndef validate_candidate(code: str, expected_function: str) -> Tuple[bool, str]:\n    try: tree = ast.parse(code)\n    except SyntaxError as exc: return False, f"syntax:{exc}"\n    if len(tree.body) != 1 or not isinstance(tree.body[0], ast.FunctionDef): return False, "one function required"\n    if tree.body[0].name != expected_function: return False, "wrong function name"\n    for node in ast.walk(tree):\n        if isinstance(node, FORBIDDEN): return False, f"forbidden:{type(node).__name__}"\n        if isinstance(node, ast.Name) and node.id.startswith("__"): return False, "dunder name"\n        if isinstance(node, ast.Attribute) and node.attr.startswith("__"): return False, "dunder attr"\n        if isinstance(node, ast.Call):\n            if isinstance(node.func, ast.Name) and node.func.id not in {"range", "len", "ValueError", "RuntimeError"}: return False, f"call:{node.func.id}"\n            if isinstance(node.func, ast.Attribute) and node.func.attr not in {"pop"}: return False, f"method:{node.func.attr}"\n    return True, "ok"\n\n\ndef replace_function(module_text: str, function_name: str, candidate: str) -> str:\n    tree = ast.parse(module_text)\n    target = next((n for n in tree.body if isinstance(n, ast.FunctionDef) and n.name == function_name), None)\n    if target is None: raise ValueError(f"function {function_name} not found")\n    lines = module_text.splitlines()\n    start = target.lineno - 1\n    end = target.end_lineno or target.lineno\n    return "\\n".join(lines[:start] + candidate.strip().splitlines() + lines[end:]) + "\\n"\n\n\ndef evaluate_in_repo(issue: Dict[str, Any], candidate: str, work_root: Path) -> Tuple[bool, str, str]:\n    candidate = normalize_code(candidate)\n    ok, reason = validate_candidate(candidate, issue["function"])\n    if not ok: return False, reason, ""\n    repo_dir = work_root / f"issue_{issue[\'seed\']}_{issue[\'id\']}_{issue[\'repo\']}"\n    if repo_dir.exists(): shutil.rmtree(repo_dir)\n    for path, text in repo_files(issue["repo"]).items():\n        target = repo_dir / path\n        target.parent.mkdir(parents=True, exist_ok=True)\n        target.write_text(text, encoding="utf-8")\n    module_path = repo_dir / issue["module"]\n    module_path.write_text(replace_function(module_path.read_text(), issue["function"], candidate), encoding="utf-8")\n    tests = repo_dir / "hidden_test.py"\n    tests.write_text(hidden_test_source(issue), encoding="utf-8")\n    env = os.environ.copy()\n    env["PYTHONPATH"] = str(repo_dir / "src")\n    try:\n        proc = subprocess.run([sys.executable, "-m", "unittest", "-q", "hidden_test.py"], cwd=repo_dir, env=env, capture_output=True, text=True, timeout=8)\n    except subprocess.TimeoutExpired: return False, "test timeout", ""\n    output = (proc.stdout + "\\n" + proc.stderr).strip()\n    return proc.returncode == 0, "pass" if proc.returncode == 0 else "hidden tests failed", output\n\n\ndef derive_repo(_stats: Dict[str, Any], _profile: Dict[str, Any], features: Dict[str, Any]):\n    if not features.get("teach"): return None\n    family = features["family"]\n    return {\n        "family": family,\n        "rule": features["rule"],\n        "decision_label": f"v{features[\'version\']}",\n        "lifecycle_state": "VERIFIED",\n        "evidence": {"count": features["evidence_count"], "version": features["version"]},\n        "admit_threshold": 2,\n        "schema_template": FAMILY_META[family]["template"],\n        "template_metadata": {"source": "repo_public_metadata_renderer", "shape_key": FAMILY_META[family]["shape"], "value_free": True},\n    }\n\n\ndef prompt_for(issue: Dict[str, Any], memory: str) -> Tuple[str, str]:\n    system = "You are editing a real Python repository. Return only JSON with string field content containing exactly one replacement function."\n    teaching = f"\\nCURRENT MAINTAINER CONVENTION:\\n{issue[\'rule\']}\\n" if issue["teach"] else ""\n    user = (\n        f"REPOSITORY: {issue[\'repo\']}\\nTARGET FILE: {issue[\'module\']}\\nTARGET FUNCTION: {issue[\'function\']}\\n"\n        + teaching\n        + "CURRENT PARAMETERS JSON:\\n" + json.dumps(issue["params"], sort_keys=True) + "\\n"\n        + "VISIBLE REPOSITORY FILES:\\n" + repo_context(issue["repo"]) + "\\n"\n        + ("ADAPTATION MEMORY:\\n" + memory + "\\n" if memory else "")\n        + "Return the replacement function. Hidden tests are not visible and must not be guessed or modified."\n    )\n    return system, user\n\n\nclass MockModel:\n    def __call__(self, _system: str, user: str, issue: Dict[str, Any]) -> Dict[str, Any]:\n        if "CURRENT_INSTRUCTION_TEMPLATE" in user or "SCHEMA_TEMPLATE" in user:\n            content = issue["expected_code"]\n        else:\n            content = f"def {issue[\'function\']}(*args):\\n    return None\\n"\n        return {"content": content, "prompt_tokens": token_count_approx(user), "generated_tokens": token_count_approx(content), "usable": True, "strict_json": True, "attempts": 1}\n\n\nclass HFModel:\n    def __init__(self, model_name: str):\n        import torch\n        from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig\n        assert torch.cuda.is_available()\n        self.torch = torch\n        dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16\n        quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=dtype)\n        self.tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)\n        if self.tokenizer.pad_token_id is None: self.tokenizer.pad_token_id = self.tokenizer.eos_token_id\n        self.model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto", quantization_config=quant, torch_dtype=dtype)\n        self.model.eval()\n    def count(self, text: str) -> int: return len(self.tokenizer.encode(text or "", add_special_tokens=False))\n    def _parse(self, raw: str) -> Tuple[Optional[str], bool]:\n        try:\n            obj = json.loads(raw.strip())\n            if isinstance(obj, dict) and isinstance(obj.get("content"), str): return obj["content"], True\n        except Exception: pass\n        decoder = json.JSONDecoder()\n        for m in re.finditer(r"\\{", raw):\n            try:\n                obj, _ = decoder.raw_decode(raw[m.start():])\n                if isinstance(obj, dict) and isinstance(obj.get("content"), str): return obj["content"], False\n            except Exception: pass\n        lines = raw.splitlines()\n        for end in range(len(lines), 0, -1):\n            candidate = "\\n".join(lines[:end]).strip()\n            try:\n                tree = ast.parse(candidate)\n                if len(tree.body) == 1 and isinstance(tree.body[0], ast.FunctionDef): return candidate, False\n            except Exception: pass\n        return None, False\n    def __call__(self, system: str, user: str, issue: Dict[str, Any]) -> Dict[str, Any]:\n        attempts = total_prompt = total_generated = 0\n        raws = []\n        for retry in range(2):\n            attempts += 1\n            current_user = user if retry == 0 else user + "\\nPrior output invalid. Return only {\\"content\\": \\"def ...\\"}."\n            prompt = self.tokenizer.apply_chat_template([{"role":"system","content":system},{"role":"user","content":current_user}], tokenize=False, add_generation_prompt=True, enable_thinking=False)\n            inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)\n            total_prompt += int(inputs["input_ids"].shape[-1])\n            with self.torch.inference_mode(): out = self.model.generate(**inputs, max_new_tokens=420, do_sample=False, pad_token_id=self.tokenizer.pad_token_id)\n            gen = out[0, inputs["input_ids"].shape[-1]:]\n            total_generated += int(gen.shape[-1])\n            raw = self.tokenizer.decode(gen, skip_special_tokens=True); raws.append(raw)\n            content, strict = self._parse(raw)\n            if content is not None: return {"content": content, "prompt_tokens": total_prompt, "generated_tokens": total_generated, "usable": True, "strict_json": strict, "attempts": attempts, "raw_outputs": raws}\n        return {"content":"", "prompt_tokens":total_prompt, "generated_tokens":total_generated, "usable":False, "strict_json":False, "attempts":attempts, "raw_outputs":raws}\n\n\ndef run_arm(issues: List[Dict[str, Any]], arm: str, model: Any, work_root: Path, icl_budget: int, akili_budget: int, fast_mock: bool = False) -> Dict[str, Any]:\n    count_fn = getattr(model, "count", token_count_approx)\n    if arm == "stateless": sysobj: Any = Stateless(count_fn)\n    elif arm == "bounded_icl": sysobj = BoundedICL(icl_budget, count_fn)\n    elif arm == "akili": sysobj = AkiliCore(akili_budget, count_fn, derive_repo, min_evidence=2, max_profiles=4, max_versions=48, recent_window=10, max_stats_keys=64, max_retrieval_log=256)\n    else: raise KeyError(arm)\n    evidence = defaultdict(int)\n    traces = []\n    active_versions: Dict[Tuple[str,str], int] = {}\n    for issue in issues:\n        if hasattr(sysobj, "switch_entity"): sysobj.switch_entity(issue["repo"])\n        memory, provenance, details = "", "NO_MEMORY", {}\n        if arm == "bounded_icl": memory, provenance = sysobj.memory_prompt(), "RAW_HISTORY"\n        elif arm == "akili":\n            if issue["teach"]: memory, provenance = current_template(issue["family"]), "CURRENT_INSTRUCTION_TEMPLATE"\n            else:\n                memory, details = sysobj.retrieval_context(issue["family"], issue["shape"])\n                provenance = details.get("primary_memory_class", "NO_MEMORY")\n        memory, memory_tokens = truncate_to_budget(memory, akili_budget if arm == "akili" else icl_budget, count_fn)\n        system_prompt, user_prompt = prompt_for(issue, memory)\n        response = model(system_prompt, user_prompt, issue)\n        if fast_mock:\n            candidate = normalize_code(response.get("content", ""))\n            ast_ok, ast_reason = validate_candidate(candidate, issue["function"])\n            passed = ast_ok and candidate.strip() == issue["expected_code"].strip()\n            reason = "pass" if passed else (ast_reason if not ast_ok else "mock hidden-test mismatch")\n            test_output = "FAST_MOCK_EQUIVALENCE"\n        else:\n            passed, reason, test_output = evaluate_in_repo(issue, response.get("content", ""), work_root / arm)\n        if arm == "bounded_icl":\n            sysobj.observe_episode(user_prompt + "\\nASSISTANT:\\n" + response.get("content", "") + f"\\nVERIFIER:{\'PASS\' if passed else \'FAIL\'}", {}, issue["repo"], "PASS" if passed else "FAIL")\n        elif arm == "akili":\n            key = (issue["repo"], issue["family"], issue["version"])\n            if issue["teach"]: evidence[key] += 1\n            sysobj.observe_episode(\n                f"{issue[\'family\']} v{issue[\'version\']}",\n                {"teach": issue["teach"], "family": issue["family"], "version": issue["version"], "rule": issue["rule"], "evidence_count": evidence[key], f"latest_{issue[\'family\']}_version": issue["version"]},\n                issue["repo"], "PASS" if passed else "FAIL",\n            )\n            if issue["teach"] and evidence[key] >= 2: active_versions[(issue["repo"], issue["family"])] = issue["version"]\n        active_label = None\n        if arm == "akili" and not issue["teach"]:\n            active = sysobj.active_record(issue["family"])\n            active_label = (active or {}).get("decision_label")\n        obsolete = bool(arm == "akili" and not issue["teach"] and active_label != f"v{issue[\'version\']}")\n        traces.append({\n            "issue_id": issue["id"], "seed": issue["seed"], "arm": arm, "repo": issue["repo"], "family": issue["family"], "version": issue["version"],\n            "teach": issue["teach"], "slice": issue["slice"], "passed": passed, "reason": reason, "test_output": test_output[-1000:],\n            "memory_provenance": provenance, "memory_tokens": memory_tokens, "prompt_tokens": response.get("prompt_tokens",0), "generated_tokens": response.get("generated_tokens",0),\n            "usable": response.get("usable",False), "strict_json": response.get("strict_json",False), "attempts": response.get("attempts",0),\n            "obsolete_memory": obsolete, "active_decision_label": active_label, "details": details,\n        })\n    return {"arm": arm, "traces": traces, "system_report": sysobj.report()}\n\n\ndef aggregate(data: Dict[int, Dict[str, Any]]) -> Dict[str, Any]:\n    out = {"arms": {}, "integrity": {}, "performance": {}}\n    for arm in ARMS:\n        rows = [t for seed in data.values() for t in seed[arm]["traces"]]\n        slices = {}\n        for name in sorted({r["slice"] for r in rows}):\n            subset = [r for r in rows if r["slice"] == name]\n            slices[name] = {"issues":len(subset), "passed":sum(r["passed"] for r in subset), "rate":sum(r["passed"] for r in subset)/len(subset)}\n        out["arms"][arm] = {\n            "issues":len(rows), "passed":sum(r["passed"] for r in rows), "accuracy":sum(r["passed"] for r in rows)/len(rows),\n            "prompt_tokens":sum(r["prompt_tokens"] for r in rows), "generated_tokens":sum(r["generated_tokens"] for r in rows),\n            "memory_tokens":sum(r["memory_tokens"] for r in rows), "retries":sum(max(0,r["attempts"]-1) for r in rows),\n            "usable_rate":sum(r["usable"] for r in rows)/len(rows), "strict_rate":sum(r["strict_json"] for r in rows)/len(rows),\n            "obsolete_retrievals":sum(r["obsolete_memory"] for r in rows), "by_slice":slices,\n        }\n    all_rows = [r for seed in data.values() for arm in ARMS for r in seed[arm]["traces"]]\n    integrity = {\n        "locked_three_seed_design": len(data)==3,\n        "equal_issue_counts": len({len(data[s][a]["traces"]) for s in data for a in ARMS})==1,\n        "repo_fixture_hashes_present": all(data[s].get("fixture_hash") for s in data),\n        "akili_teaching_template_exact": all(r["memory_provenance"]=="CURRENT_INSTRUCTION_TEMPLATE" for r in all_rows if r["arm"]=="akili" and r["teach"]),\n        "akili_recall_persistent": all(r["memory_provenance"]=="PERSISTENT_SCHEMA_TEMPLATE" for r in all_rows if r["arm"]=="akili" and not r["teach"]),\n        "obsolete_retrieval_zero": all(not r["obsolete_memory"] for r in all_rows if r["arm"]=="akili"),\n        "cross_scope_zero": all(data[s]["akili"]["system_report"].get("cross_scope_retrievals",0)==0 for s in data),\n        "audits_valid": all(data[s]["akili"]["system_report"].get("audit_valid",False) for s in data),\n        "states_bounded": all(data[s][a]["system_report"].get("state_bounded",True) for s in data for a in ARMS),\n    }\n    out["integrity"] = integrity\n    ak, icl = out["arms"]["akili"], out["arms"]["bounded_icl"]\n    reduction = 1-ak["prompt_tokens"]/max(icl["prompt_tokens"],1)\n    criteria = {\n        "akili_accuracy_at_least_90": ak["accuracy"]>=0.90,\n        "akili_within_5pp_icl": ak["accuracy"]+0.05>=icl["accuracy"],\n        "return_after_switch_at_least_90": ak["by_slice"]["return_after_repo_switch"]["rate"]>=0.90,\n        "post_supersession_at_least_90": ak["by_slice"]["post_supersession_return"]["rate"]>=0.90,\n        "prompt_reduction_at_least_60": reduction>=0.60,\n        "all_integrity_pass": all(integrity.values()),\n    }\n    out["performance"] = {"criteria":criteria, "prompt_reduction":reduction, "PASS":all(criteria.values())}\n    return out\n\n\ndef run(phase: str, mode: str, output: Path, model_name: str, icl_budget: int, akili_budget: int) -> Dict[str, Any]:\n    seeds = LOCKED_PHASE_SEEDS[phase]\n    output = output.expanduser().resolve()\n    output.mkdir(parents=True, exist_ok=True)\n    work_root = output / "work"; work_root.mkdir(exist_ok=True)\n    model: Any = MockModel() if mode=="mock" else HFModel(model_name)\n    data: Dict[int, Dict[str, Any]] = {}\n    fixture_hash = canonical_hash({repo:repo_files(repo) for repo in ("solar_ops","field_data")})\n    tests_hash = canonical_hash({s:[hidden_test_source(i) for i in make_issues(s)] for s in seeds})\n    execution_fixture = []\n    if mode == "mock":\n        fixture_root = work_root / "execution_fixture"\n        for issue in make_issues(seeds[0]):\n            if issue["slice"] == "acquisition_1":\n                ok, reason, output_text = evaluate_in_repo(issue, issue["expected_code"], fixture_root)\n                execution_fixture.append({"family": issue["family"], "passed": ok, "reason": reason})\n        assert execution_fixture and all(row["passed"] for row in execution_fixture), execution_fixture\n    for seed in seeds:\n        issues = make_issues(seed)\n        data[seed] = {"fixture_hash":fixture_hash, "tests_hash":tests_hash}\n        for arm in ARMS:\n            print(f"seed={seed} arm={arm}", flush=True)\n            data[seed][arm] = run_arm(issues, arm, model, work_root/f"seed{seed}", icl_budget, akili_budget, fast_mock=(mode=="mock"))\n            (output/f"seed{seed}_{arm}.json").write_text(json.dumps(data[seed][arm],indent=2),encoding="utf-8")\n    summary = aggregate(data)\n    receipt = {"protocol":PROTOCOL,"phase":phase,"mode":mode,"seeds":seeds,"model":model_name if mode=="real" else "prompt-path mock","fixture_hash":fixture_hash,"hidden_tests_hash":tests_hash,"execution_fixture":execution_fixture,"summary":summary,"script_sha256":sha256_bytes(Path(__file__).read_bytes())}\n    (output/"FINAL_RECEIPT.json").write_text(json.dumps(receipt,indent=2),encoding="utf-8")\n    print(json.dumps(summary,indent=2))\n    return receipt\n\n\ndef parse_args(argv: Optional[Iterable[str]]=None) -> argparse.Namespace:\n    p=argparse.ArgumentParser()\n    p.add_argument("--phase",choices=LOCKED_PHASE_SEEDS,default=os.getenv("AKILI_REPO_PHASE","development"))\n    p.add_argument("--mode",choices=["mock","real"],default=os.getenv("AKILI_REPO_MODE","mock"))\n    p.add_argument("--output",type=Path,default=Path(os.getenv("AKILI_REPO_OUTPUT","./akili_repo_output")))\n    p.add_argument("--model",default=os.getenv("AKILI_REPO_MODEL","Qwen/Qwen3-4B"))\n    p.add_argument("--icl-budget",type=int,default=int(os.getenv("AKILI_REPO_ICL_BUDGET","1400")))\n    p.add_argument("--akili-budget",type=int,default=int(os.getenv("AKILI_REPO_AKILI_BUDGET","320")))\n    return p.parse_args(argv)\n\nif __name__=="__main__":\n    a=parse_args(); receipt=run(a.phase,a.mode,a.output,a.model,a.icl_budget,a.akili_budget)\n    raise SystemExit(0 if all(receipt["summary"]["integrity"].values()) else 2)\n'
script_path = WORK_ROOT / "real_repo_coding_agent.py"
script_path.write_text(BENCHMARK_SOURCE, encoding="utf-8")
subprocess.run([sys.executable, "-m", "py_compile", str(core_path), str(script_path)], check=True)
print("script sha256:", hashlib.sha256(script_path.read_bytes()).hexdigest())


In [ ]:
# Mandatory offline/prompt-path preflight. This consumes no Qwen inference.
mock_out = WORK_ROOT / "mock_preflight"
subprocess.run([
    sys.executable, str(script_path), "--phase", "development", "--mode", "mock",
    "--output", str(mock_out)
], check=True, cwd=WORK_ROOT, timeout=300)
mock_receipt = json.loads((mock_out / "FINAL_RECEIPT.json").read_text())
assert all(mock_receipt["summary"]["integrity"].values()), mock_receipt["summary"]["integrity"]
print("MOCK PREFLIGHT PASS")


In [ ]:
# REAL QWEN RUN — phase locked. Do not edit seeds, prompts, templates or scores after starting.
cmd = [
    sys.executable, str(script_path), "--phase", PHASE, "--mode", "real",
    "--output", str(OUT), "--model", MODEL
]
print("RUNNING:", " ".join(cmd), flush=True)
subprocess.run(cmd, check=True, cwd=WORK_ROOT)
receipt = json.loads((OUT / "FINAL_RECEIPT.json").read_text())
print(json.dumps(receipt, indent=2))
